# 05 · Del híbrido al despliegue: permisos, abstención, regresiones y latencia

**Idea:** un híbrido que funciona en el notebook todavía no es un sistema productivo. Aquí cubres **cuatro controles** que
los tutoriales omiten (controles de producción; al final tienes la lista de verificación):

1. **Permisos (ACL)** aplicados *antes* de rankear: nadie recupera lo que no le toca.
2. **Abstención**: si no hay evidencia suficiente, responder «no lo sé» en vez de inventar (patrón *corrective RAG*).
3. **Puerta de regresión**: un cambio que empeora la recuperación no llega a producción.
4. **Latencia**: medir p50 y p95 de cada método.

## Glosario · producción

Lo que necesitas para leer este notebook (puedes volver aquí cuando una palabra no te suene):

- **Embedding:** lista de números que representa el *significado* de un texto; textos con sentido parecido quedan «cerca», aunque no compartan palabras.
- **Similitud coseno:** qué tan cerca están dos embeddings (cerca de 1 = parecidos; cerca de 0 = nada que ver).
- **BM25:** puntaje por *palabras exactas*: cuenta qué documentos contienen las palabras de la pregunta y valora más las raras (`ERR-5012`) que las comunes. No entiende sinónimos; si no comparte ninguna palabra, no devuelve nada. Su puntaje no tiene tope y no es comparable con el coseno.
- **Top-k:** los *k* primeros resultados del ranking (top-3 = los tres mejores).
- **Híbrido calibrado:** mezcla `α · coseno + (1 − α) · BM25` (ambos llevados a 0–1), con α elegido en el set de ajuste. Se explica en el notebook 01; aquí se reutiliza.
- **MRR y golden set, ajuste y prueba:** ver los notebooks 01 (MRR = promedio de 1 ÷ posición del primer correcto; ajuste y prueba son los dos conjuntos de preguntas).
- **ACL** (*Access Control List*): lista de quién puede ver qué. Los documentos no permitidos se descartan **antes** de ordenar.
- **Abstención (*corrective RAG*):** responder «no lo sé» cuando la evidencia es floja, en vez de inventar.
- **τ (tau) y β (beta):** umbrales de abstención. El sistema se abstiene solo si el mejor coseno es menor que τ **y** el mejor BM25 es menor que β.
- **Puerta de regresión:** un cambio pasa solo si `MRR_candidato ≥ MRR_vigente − tolerancia` (aquí, tolerancia 0.03), medido en el set de prueba.
- **CI** (*integración continua*): pruebas automáticas que se corren con cada cambio; la puerta de regresión vive ahí.
- **Latencia p50 / p95:** tiempo en que responde el 50 % / el 95 % de las consultas. **SLA:** compromiso de tiempo de respuesta con el usuario.

## Los datos de este notebook (qué es cada campo)

| Conjunto | Qué es | Campos de cada elemento |
|---|---|---|
| `DOCS` y `QUERIES` | los mismos 42 documentos y 33 preguntas del notebook 01 | `(id, título, texto, tipo)` y `(pregunta, tipo, relevantes)` |
| `RESTRINGIDOS` y `ROLES_CONTRATOS` | qué documentos son confidenciales (contratos) y qué roles pueden verlos | conjuntos de id y de nombres de rol |
| `FUERA` | 12 preguntas ajenas al dominio (ceviche, Tokio…) | lista de textos; las pares van a ajuste y las impares a prueba |
| `CASI` | 4 preguntas «casi del dominio» cuyo dato no existe | lista de textos; es la prueba difícil de la abstención |

In [1]:
import importlib.util, subprocess, sys

for paquete, modulo in [("rank-bm25", "rank_bm25"), ("sentence-transformers", "sentence_transformers")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", paquete])
print("Entorno listo")

Entorno listo


In [2]:
#@title Datos sintéticos de «Andina Retail» (no necesitas leer esta celda) { display-mode: "form" }
"""Corpus sintético de «Andina Retail» para los notebooks del taller.

Se inserta tal cual en cada notebook (celda oculta con #@title), así cada uno corre solo en Colab.
Todo es inventado: las métricas que salgan ilustran un criterio, no miden producción.
"""

# ---------------------------------------------------------------- documentos
# ESQUEMA · DOCS (el corpus: 42 «documentos» cortos, cada uno es una tupla de 4 campos)
#   (id, título, texto, tipo)
#   id     código único del documento. La letra dice de qué familia es:
#            P = política interna · T = técnico (código exacto) · D = «distractor» (código casi idéntico a uno T)
#            E = entidades y relaciones (personas, equipos, sistemas; sirven para el notebook de grafo)
#   título nombre corto del documento (se indexa junto con el texto).
#   texto  contenido que se busca y que se le entregaría al modelo (1–3 frases).
#   tipo   categoría temática: politica · tecnico · producto · contrato · organizacion · incidente.
#          No se usa para buscar; solo describe el documento.
# Ejemplo: ("P01", "Vacaciones", "Los colaboradores acumulan 30 días…", "politica")
DOCS = [
    # Políticas (preguntas semánticas, con paráfrasis)
    ("P01", "Vacaciones", "Los colaboradores acumulan 30 días calendario de descanso remunerado por año de servicio. El periodo puede fraccionarse en bloques de mínimo 7 días con aprobación del jefe directo.", "politica"),
    ("P02", "Teletrabajo", "Se permite el trabajo remoto hasta tres días por semana. El día presencial obligatorio es el martes. Se requiere conexión VPN corporativa y equipo entregado por la empresa.", "politica"),
    ("P03", "Devoluciones", "El cliente puede devolver un producto dentro de 30 días desde la compra, con boleta y empaque sellado. Los productos de electrónica tienen un plazo de 15 días. El reembolso se realiza al mismo medio de pago.", "politica"),
    ("P04", "Viáticos", "Los gastos de viaje se reembolsan con comprobante dentro de 10 días hábiles. El tope diario de alimentación es 120 soles. Los viajes internacionales requieren aprobación de la gerencia del área.", "politica"),
    ("P05", "Licencias", "La licencia por paternidad es de 10 días calendario y la de maternidad de 98 días. Se solicita a Recursos Humanos con el certificado médico correspondiente.", "politica"),
    ("P06", "Seguridad de la información", "Las claves de acceso deben tener 14 caracteres y cambiarse cada 90 días. Todo acceso a sistemas internos exige verificación en dos pasos (MFA). Está prohibido compartir credenciales.", "politica"),
    ("P07", "Código de conducta: obsequios", "No se aceptan obsequios, invitaciones ni atenciones de proveedores con valor mayor a 100 soles. Todo ofrecimiento debe reportarse al área de Cumplimiento.", "politica"),
    ("P08", "Proceso de compras", "Toda compra sobre 5,000 soles exige tres cotizaciones comparables y la aprobación del comité de compras. Las compras menores se tramitan con una sola cotización.", "politica"),
    ("P09", "Capacitación", "Cada colaborador dispone de un presupuesto anual de 2,000 soles para cursos, certificaciones y congresos, previa autorización de su jefe directo.", "politica"),
    ("P10", "Horario y tardanzas", "El horario de oficina es de 9:00 a 18:00. Las tardanzas acumuladas superiores a 60 minutos en el mes se descuentan de la remuneración y tres tardanzas generan una amonestación escrita.", "politica"),
    # Técnicos con códigos exactos (palabra clave)
    ("T01", "ERR-5021 pasarela de pagos", "ERR-5021: fallo de conexión con la pasarela de pagos PayHub. Reintentar tras 30 segundos; si persiste, escalar al equipo Pagos con el identificador de la transacción.", "tecnico"),
    ("T02", "ERR-7714 inventario", "ERR-7714: inventario desincronizado entre la tienda y el almacén central. Ejecutar la reconciliación nocturna con el job REC-NIGHT-02 y validar el stock al día siguiente.", "tecnico"),
    ("T03", "ERR-1003 sesión", "ERR-1003: el token de sesión de la aplicación móvil expiró. El usuario debe volver a iniciar sesión; no requiere escalamiento.", "tecnico"),
    ("T04", "SKU-48213 licuadora", "SKU-48213: Licuadora Oster ProBlend de 1.5 litros, línea hogar. Garantía de 24 meses con boleta de compra.", "producto"),
    ("T05", "SKU-90117 laptop", "SKU-90117: Laptop Lenovo IdeaPad 5 con 16 GB de RAM, línea electrónica. Garantía de 12 meses con boleta de compra.", "producto"),
    ("T06", "SKU-33502 refrigeradora", "SKU-33502: Refrigeradora Samsung de 380 litros con tecnología inverter, línea blanca. Garantía de 36 meses con boleta de compra.", "producto"),
    ("T07", "RB-204 certificados TLS", "RB-204: procedimiento para rotar los certificados TLS del balanceador. Se ejecuta en ventana de mantenimiento, con aprobación del responsable de Infraestructura y validación posterior de los endpoints.", "tecnico"),
    ("T08", "Contrato CT-2024-0087", "CT-2024-0087: contrato de transporte con el proveedor Logística Andes, vigente hasta diciembre de 2026, con penalidad del 2% por entregas tardías.", "contrato"),
    # Distractores: códigos casi idénticos, como en cualquier catálogo o log real
    ("D01", "ERR-5012 pasarela", "ERR-5012: tiempo de espera agotado al consultar el estado de un pago en la pasarela de pagos PayHub. Reintentar la consulta una vez y validar la transacción manualmente.", "tecnico"),
    ("D02", "ERR-5201 conciliación", "ERR-5201: diferencia en la conciliación bancaria diaria de la pasarela de pagos. Revisar el reporte de conciliación y escalar al equipo de Finanzas Digital.", "tecnico"),
    ("D03", "ERR-7741 stock", "ERR-7741: el stock reservado de un pedido supera el inventario disponible en tienda. Liberar la reserva y volver a calcular la disponibilidad.", "tecnico"),
    ("D04", "ERR-1030 sesión web", "ERR-1030: la sesión del portal web de clientes fue cerrada por inactividad. El usuario debe iniciar sesión nuevamente.", "tecnico"),
    ("D05", "SKU-48231 licuadora", "SKU-48231: Licuadora Oster ProBlend de 1.2 litros, línea hogar. Garantía de 12 meses con boleta de compra.", "producto"),
    ("D06", "SKU-48123 batidora", "SKU-48123: Batidora de pedestal Oster de 5 velocidades, línea hogar. Garantía de 18 meses con boleta de compra.", "producto"),
    ("D07", "SKU-90171 laptop", "SKU-90171: Laptop Lenovo IdeaPad 3 con 8 GB de RAM, línea electrónica. Garantía de 12 meses con boleta de compra.", "producto"),
    ("D08", "SKU-33205 refrigeradora", "SKU-33205: Refrigeradora Samsung de 300 litros, línea blanca. Garantía de 24 meses con boleta de compra.", "producto"),
    ("D09", "RB-240 certificados", "RB-240: procedimiento para renovar los certificados del correo corporativo. Se ejecuta fuera del horario de oficina y lo valida el equipo de Soporte.", "tecnico"),
    ("D10", "Contrato CT-2024-0078", "CT-2024-0078: contrato de limpieza con el proveedor Servi Limpio, vigente hasta junio de 2027, con penalidad del 1% por incumplimiento.", "contrato"),
    ("D11", "Contrato CT-2023-0087", "CT-2023-0087: contrato de vigilancia con el proveedor Seguridad Altiplano, vigente hasta marzo de 2026, renovación pendiente.", "contrato"),
    ("D12", "Job REC-NIGHT-01", "REC-NIGHT-01: job de reconciliación nocturna de ventas entre tiendas y contabilidad. Se ejecuta a las 02:00 y envía un reporte al área de Finanzas.", "tecnico"),
    # Entidades y relaciones (multi-hop)
    ("E01", "Equipo Pagos", "El equipo Pagos, liderado por Rosa Quispe, mantiene la pasarela de pagos PayHub.", "organizacion"),
    ("E02", "Equipo Finanzas Digital", "El sistema de facturación FacturaPlus es mantenido por el equipo Finanzas Digital, liderado por Marco Salas.", "organizacion"),
    ("E03", "Marco Salas", "Marco Salas reporta a la gerente de Tecnología, Lucía Paredes.", "organizacion"),
    ("E04", "Rosa Quispe", "Rosa Quispe reporta a la gerente de Tecnología, Lucía Paredes.", "organizacion"),
    ("E05", "Lucía Paredes", "Lucía Paredes, gerente de Tecnología, aprueba los viajes internacionales del área de Tecnología.", "organizacion"),
    ("E06", "PayHub y antifraude", "PayHub depende del servicio antifraude RiskGuard, operado por el proveedor externo SafeNet Perú.", "organizacion"),
    ("E07", "FacturaPlus y SUNAT", "FacturaPlus emite los comprobantes electrónicos a través del operador FactuRed, certificado ante SUNAT.", "organizacion"),
    ("E08", "Cobertura del contrato de transporte", "El contrato CT-2024-0087 con Logística Andes cubre el transporte hacia las tiendas de Arequipa y Cusco.", "organizacion"),
    ("E09", "Gestión de Logística Andes", "El proveedor Logística Andes es gestionado por el área de Compras, a cargo de Daniela Ríos.", "organizacion"),
    ("E10", "Daniela Ríos", "Daniela Ríos reporta al gerente de Operaciones, Jorge Medina.", "organizacion"),
    ("E11", "Tienda Arequipa", "La tienda de Arequipa es administrada por Carmen Valdivia, quien reporta a Jorge Medina.", "organizacion"),
    ("E12", "Incidente 3 de septiembre", "El 3 de septiembre una caída del servicio RiskGuard provocó una ráfaga de errores ERR-5021 en PayHub durante 40 minutos.", "incidente"),
]

# ------------------------------------------------- consultas con juez (doc-level)
# tipo: semantica (paráfrasis), exacta (código/ID), mixta (ambas), multihop (necesita varios docs)
# ESQUEMA · QUERIES (las 33 preguntas con respuesta conocida: el «golden set»; tuplas de 3 campos)
#   (pregunta, tipo, relevantes)
#   pregunta   texto tal como lo escribiría un usuario.
#   tipo       semantica (parafrasea; casi no comparte palabras con el documento) ·
#              exacta (un código o identificador) · mixta (lenguaje natural + código).
#   relevantes conjunto con los id de los documentos correctos. Es el «juez»: una pregunta acierta si alguno
#              de ellos aparece arriba en el ranking.
# Ejemplo: ("¿Puedo trabajar desde mi casa?", "semantica", {"P02"})  → el correcto es «Teletrabajo».
# Más abajo, dividir() parte estas 33 en 17 de ajuste (dev) y 16 de prueba (test).
QUERIES = [
    # semánticas: casi sin palabras en común con el documento
    ("¿Cuántos días libres pagados tengo cada año?", "semantica", {"P01"}),
    ("¿Puedo trabajar desde mi casa?", "semantica", {"P02"}),
    ("Compré una tele y no me gustó, ¿me devuelven la plata?", "semantica", {"P03"}),
    ("¿Me pagan lo que gasto cuando viajo por trabajo?", "semantica", {"P04"}),
    ("¿Cómo protejo mis accesos a los sistemas?", "semantica", {"P06"}),
    ("¿Puedo recibir un regalo de un proveedor?", "semantica", {"P07"}),
    ("¿Hay dinero para hacer cursos y certificaciones?", "semantica", {"P09"}),
    ("Llegué tarde varias veces, ¿qué consecuencias tiene?", "semantica", {"P10"}),
    ("¿Cuál es el plazo para cambiar un producto que no me sirvió?", "semantica", {"P03"}),
    ("¿Qué trámite hago para pedir descanso cuando nace mi hijo?", "semantica", {"P05"}),
    ("¿Cuántas ofertas de precio necesito antes de comprar algo caro?", "semantica", {"P08"}),
    ("¿A qué hora debo estar en la oficina?", "semantica", {"P10"}),
    # exactas: un código o identificador
    ("ERR-5021", "exacta", {"T01", "E12"}),
    ("SKU-48213", "exacta", {"T04"}),
    ("¿Qué significa ERR-7714?", "exacta", {"T02"}),
    ("CT-2024-0087", "exacta", {"T08", "E08"}),
    ("RB-204", "exacta", {"T07"}),
    ("REC-NIGHT-02", "exacta", {"T02"}),
    ("SKU-33502", "exacta", {"T06"}),
    ("ERR-1003", "exacta", {"T03"}),
    ("SKU-90117", "exacta", {"T05"}),
    ("ERR-5012", "exacta", {"D01"}),
    ("CT-2023-0087", "exacta", {"D11"}),
    ("SKU-48231", "exacta", {"D05"}),
    # mixtas: lenguaje natural + código
    ("¿Cuántos meses de garantía tiene la laptop SKU-90117?", "mixta", {"T05"}),
    ("Mi app del celular muestra ERR-1003, ¿qué hago?", "mixta", {"T03"}),
    ("Falla la pasarela de pagos con ERR-5021, ¿cómo lo resuelvo?", "mixta", {"T01"}),
    ("¿Hasta cuándo está vigente el contrato CT-2024-0087 de transporte?", "mixta", {"T08"}),
    ("Se desincronizó el inventario con ERR-7714, ¿qué proceso corre de noche?", "mixta", {"T02"}),
    ("¿Qué garantía tiene la batidora SKU-48123?", "mixta", {"D06"}),
    ("El portal web me cerró la sesión con ERR-1030, ¿es normal?", "mixta", {"D04"}),
    ("¿De qué trata el contrato de limpieza CT-2024-0078?", "mixta", {"D10"}),
    ("Hay diferencia en la conciliación bancaria con ERR-5201, ¿a quién escalo?", "mixta", {"D02"}),
]

# Preguntas multi-hop: docs que hay que juntar para poder responder + respuesta esperada
# ESQUEMA · MULTIHOP (6 preguntas que necesitan juntar varios documentos; tuplas de 3 campos)
#   (pregunta, documentos_necesarios, respuesta_esperada)
#   documentos_necesarios  id de TODOS los documentos E## que hay que unir para poder responder.
#   respuesta_esperada     dato final que sale de encadenar esos documentos.
# Aquí la pregunta NOMBRA las entidades que hay que encadenar (el texto suele bastar).
MULTIHOP = [
    ("¿Quién aprueba los viajes internacionales de la jefa del equipo que mantiene la pasarela de pagos?",
     {"E01", "E04", "E05"}, "Lucía Paredes"),
    ("¿Qué proveedor externo opera el servicio antifraude del que depende el sistema que mantiene el equipo de Rosa Quispe?",
     {"E01", "E06"}, "SafeNet Perú"),
    ("¿A quién reporta la persona que gestiona el proveedor que transporta mercadería a Arequipa?",
     {"E08", "E09", "E10"}, "Jorge Medina"),
    ("¿Quién es la jefa de la persona que lidera el equipo que mantiene el sistema de facturación?",
     {"E02", "E03"}, "Lucía Paredes"),
    ("¿Qué servicio antifraude provocó los errores ERR-5021 del 3 de septiembre?",
     {"E12", "E06"}, "RiskGuard"),
    ("¿Quién administra la tienda cuyo gerente de Operaciones también supervisa a quien gestiona Logística Andes?",
     {"E09", "E10", "E11"}, "Carmen Valdivia"),
]

# Puente oculto: el documento intermedio casi no comparte palabras con la pregunta
# ESQUEMA · MULTIHOP_PUENTE (3 preguntas; mismo formato que MULTIHOP)
# Diferencia: el documento intermedio («puente») casi no comparte palabras con la pregunta; por eso el texto
# se lo pierde y el grafo no.
MULTIHOP_PUENTE = [
    ("¿Quién tiene que autorizar el viaje internacional del líder del equipo de facturación?",
     {"E02", "E03", "E05"}, "Lucía Paredes"),
    ("¿Quién lidera el equipo que mantiene el sistema afectado el 3 de septiembre?",
     {"E12", "E01"}, "Rosa Quispe"),
    ("¿Quién es el superior de la persona que negocia con el proveedor que transporta a Cusco?",
     {"E08", "E09", "E10"}, "Jorge Medina"),
]

# Tripletas (sujeto, relación, objeto, doc_fuente). En un caso real las extrae un LLM de los documentos.
# ESQUEMA · TRIPLES (el grafo escrito a mano: una arista por tupla de 4 campos)
#   (sujeto, relación, objeto, doc_fuente)
#   sujeto y objeto  entidades (nodos del grafo): personas, equipos, sistemas, proveedores, lugares.
#   relación         verbo que las une (lidera, reporta_a, depende_de…).
#   doc_fuente       id del documento E## de donde sale esa relación (para citar cada salto).
# Ejemplo: ("Rosa Quispe", "reporta_a", "Lucía Paredes", "E04"). En un caso real las extrae un LLM.
TRIPLES = [
    ("Equipo Pagos", "lidera", "Rosa Quispe", "E01"),
    ("Equipo Pagos", "mantiene", "PayHub", "E01"),
    ("Equipo Finanzas Digital", "lidera", "Marco Salas", "E02"),
    ("Equipo Finanzas Digital", "mantiene", "FacturaPlus", "E02"),
    ("Marco Salas", "reporta_a", "Lucía Paredes", "E03"),
    ("Rosa Quispe", "reporta_a", "Lucía Paredes", "E04"),
    ("Lucía Paredes", "aprueba_viajes_de", "área de Tecnología", "E05"),
    ("PayHub", "depende_de", "RiskGuard", "E06"),
    ("RiskGuard", "operado_por", "SafeNet Perú", "E06"),
    ("FacturaPlus", "emite_vía", "FactuRed", "E07"),
    ("CT-2024-0087", "con_proveedor", "Logística Andes", "E08"),
    ("Logística Andes", "transporta_a", "Arequipa", "E08"),
    ("Logística Andes", "gestionado_por", "Daniela Ríos", "E09"),
    ("Daniela Ríos", "reporta_a", "Jorge Medina", "E10"),
    ("Carmen Valdivia", "administra", "tienda de Arequipa", "E11"),
    ("Carmen Valdivia", "reporta_a", "Jorge Medina", "E11"),
    ("RiskGuard", "causó_caída", "ERR-5021", "E12"),
    ("ERR-5021", "ocurrió_en", "PayHub", "E12"),
    ("Logística Andes", "transporta_a", "Cusco", "E08"),
]


# ESQUEMA · tabla de ventas (la crea build_sales(); una fila por tienda × categoría × mes = 27 filas)
#   tienda (Lima · Arequipa · Cusco) · categoria (hogar · electrónica · línea blanca) · mes (julio · agosto · septiembre)
#   unidades  cantidad vendida (número aleatorio fijo, semilla 7)
#   monto     ingresos en soles = unidades × precio de la categoría
def build_sales():
    """Tabla de ventas sintética y determinista, para el caso de SQL."""
    import numpy as np
    import pandas as pd
    rng = np.random.default_rng(7)
    filas = []
    for tienda in ["Lima", "Arequipa", "Cusco"]:
        for categoria in ["hogar", "electrónica", "línea blanca"]:
            for mes in ["julio", "agosto", "septiembre"]:
                unidades = int(rng.integers(40, 400))
                precio = {"hogar": 180, "electrónica": 2400, "línea blanca": 1900}[categoria]
                filas.append((tienda, categoria, mes, unidades, unidades * precio))
    return pd.DataFrame(filas, columns=["tienda", "categoria", "mes", "unidades", "monto"])

In [3]:
#@title Herramientas de búsqueda y evaluación (puedes dejarla cerrada) { display-mode: "form" }
import re, time, warnings
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

STOP = set("""a al algo ante con como cada de del el ella en es esta este la las lo los me mi mis o para pero por
que qué se si su sus tengo tiene un una y ya cuántos cuántas cuándo cuál hay puedo hacer""".split())


def tok(texto):
    """Tokeniza en minúsculas, conserva códigos como ERR-5021 y quita palabras vacías."""
    return [w for w in re.findall(r"[\wáéíóúñ]+(?:-[\w]+)*", texto.lower()) if w not in STOP]


IDS = [d[0] for d in DOCS]
TEXTOS = [f"{titulo}. {texto}" for _, titulo, texto, _ in DOCS]
bm25 = BM25Okapi([tok(t) for t in TEXTOS])
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
EMB = modelo.encode(TEXTOS, normalize_embeddings=True)


def s_denso(q):
    """Similitud coseno entre la pregunta y cada documento."""
    return EMB @ modelo.encode([q], normalize_embeddings=True)[0]


def s_bm25(q):
    """Puntaje léxico BM25 de la pregunta contra cada documento."""
    return bm25.get_scores(tok(q))


def ranking(scores):
    return [IDS[i] for i in np.argsort(-scores, kind="stable")]  # desempate estable: mismo resultado en cualquier entorno


def r_denso(q):
    return ranking(s_denso(q))


def r_bm25(q):
    """Solo los documentos que comparten algún término con la pregunta, como hace un buscador real.
    (Los de puntaje 0 no son «resultados»: ordenarlos solo agrega ruido arbitrario.)"""
    s = s_bm25(q)
    return [IDS[i] for i in np.argsort(-s, kind="stable") if s[i] > 0]


def minmax(x):
    x = np.asarray(x, float)
    return (x - x.min()) / (x.max() - x.min() + 1e-9)


def convexa(q, alpha):
    """Híbrido por combinación de puntajes normalizados. alpha=1 es solo denso; alpha=0 es solo BM25."""
    return ranking(alpha * minmax(s_denso(q)) + (1 - alpha) * minmax(s_bm25(q)))


def rrf(rankings, pesos=None, k=60):
    """Reciprocal Rank Fusion: fusiona listas ordenadas sin necesidad de normalizar puntajes."""
    pesos = pesos or [1] * len(rankings)
    acum = {}
    for lista, w in zip(rankings, pesos):
        for pos, doc in enumerate(lista):
            acum[doc] = acum.get(doc, 0) + w / (k + pos + 1)
    return sorted(acum, key=lambda d: -acum[d])


def evaluar(fn, consultas):
    """hit@3 (¿aparece un relevante entre los 3 primeros?) y MRR (1/posición del primer relevante), por tipo."""
    filas = []
    for q, tipo, relevantes in consultas:
        r = fn(q)
        pos = next((i + 1 for i, d in enumerate(r) if d in relevantes), None)   # None = no apareció
        filas.append((tipo, float(pos is not None and pos <= 3), 1 / pos if pos else 0.0))
    df = pd.DataFrame(filas, columns=["tipo", "hit@3", "MRR"])
    out = df.groupby("tipo").mean()
    out.loc["PROMEDIO TIPOS"] = out.mean()
    return out.round(2)


def dividir(consultas):
    """Separa cada tipo de consulta alternando: la mitad para ajustar (dev) y la otra para probar (test)."""
    cont, dev, test = {}, [], []
    for c in consultas:
        i = cont.get(c[1], 0)
        cont[c[1]] = i + 1
        (dev if i % 2 == 0 else test).append(c)
    return dev, test


print(f"{len(DOCS)} documentos · {len(QUERIES)} consultas con juez · modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2")

42 documentos · 33 consultas con juez · modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2


## 1. Permisos antes del ranking

Los contratos son confidenciales: solo ven `compras` y `legal`. Si filtras *después* de generar la respuesta, el contenido ya pasó por el modelo.
Aquí el filtro se aplica a los puntajes **antes** de ordenar.

In [4]:
RESTRINGIDOS = {"T08", "D10", "D11", "E08"}   # contratos y su cobertura
ROLES_CONTRATOS = {"compras", "legal"}


def visibles(rol):
    return np.array([d not in RESTRINGIDOS or rol in ROLES_CONTRATOS for d in IDS])


def buscar_con_acl(q, rol, alpha=0.7, k=3):
    s = alpha * minmax(s_denso(q)) + (1 - alpha) * minmax(s_bm25(q))
    s = np.where(visibles(rol), s, -np.inf)          # el documento prohibido ni siquiera compite
    return [IDS[i] for i in np.argsort(-s)[:k] if np.isfinite(s[i])]


q = "CT-2024-0087 vigencia"
print("sin filtro (practicante):", convexa(q, 0.7)[:3])
print("con ACL    (practicante):", buscar_con_acl(q, "practicante"))
print("con ACL    (compras)    :", buscar_con_acl(q, "compras"))

sin filtro (practicante): ['T08', 'D11', 'E08']
con ACL    (practicante): ['T07', 'T02', 'D06']
con ACL    (compras)    : ['T08', 'D11', 'E08']


In [5]:
# Prueba de fuga: ninguna consulta del set puede devolverle un documento restringido a un rol sin permiso
fugas = sum(len(set(buscar_con_acl(q, "practicante", k=10)) & RESTRINGIDOS) for q, _, _ in QUERIES)
print("Documentos restringidos devueltos a un practicante en", len(QUERIES), "consultas:", fugas)
assert fugas == 0

Documentos restringidos devueltos a un practicante en 33 consultas: 0


## 2. Abstención: «no lo sé» también es una respuesta correcta

Mezclamos consultas del corpus con **preguntas fuera de alcance**. Se abstiene si la mejor similitud densa **y** el mejor BM25 son bajos.
Los umbrales se ajustan en `dev` y se informan en `test`.

In [6]:
# ESQUEMA · FUERA: lista de preguntas AJENAS al dominio (ceviche, Tokio…); el sistema debería abstenerse.
# Se reparten por mitades: las de índice par van a ajuste y las impares a prueba.
FUERA = [
    "¿Cuál es la capital de Australia?", "¿Cómo preparo un ceviche?", "¿Quién ganó el mundial de 2014?",
    "Receta de lomo saltado", "¿Cuánto cuesta un bitcoin hoy?", "¿Cómo se programa en Rust?",
    "¿Qué clima hará mañana en Tokio?", "Cuéntame un chiste", "¿Cuántos habitantes tiene Brasil?",
    "¿Quién escribió Cien años de soledad?", "¿Cómo funciona un motor de combustión?", "¿Cuál es la distancia a la Luna?",
]
dev_in, test_in = dividir(QUERIES)
dev_out, test_out = FUERA[0::2], FUERA[1::2]


def señales(q):
    return float(s_denso(q).max()), float(s_bm25(q).max())


def exactitud_abstencion(consultas_in, consultas_out, tau, beta):
    """Acierta si responde lo que está en el corpus y se abstiene con lo que no."""
    ok_in = [not (d < tau and b < beta) for d, b in map(señales, [q for q, _, _ in consultas_in])]
    ok_out = [(d < tau and b < beta) for d, b in map(señales, consultas_out)]
    return np.mean(ok_in + ok_out), np.mean(ok_in), np.mean(ok_out)


mejor = None
for tau in np.round(np.arange(0.2, 0.8, 0.05), 2):
    for beta in np.round(np.arange(1, 12, 1.0), 1):
        e, _, _ = exactitud_abstencion(dev_in, dev_out, tau, beta)
        if mejor is None or e > mejor[0]:
            mejor = (e, tau, beta)
print(f"Umbrales elegidos en dev: τ(denso)={mejor[1]} · β(BM25)={mejor[2]}  (exactitud dev {mejor[0]:.2f})")

e, resp, abst = exactitud_abstencion(test_in, test_out, mejor[1], mejor[2])
print(f"TEST → exactitud {e:.2f} · responde lo que debe {resp:.2f} · se abstiene cuando debe {abst:.2f}")

Umbrales elegidos en dev: τ(denso)=0.4 · β(BM25)=1.0  (exactitud dev 1.00)
TEST → exactitud 1.00 · responde lo que debe 1.00 · se abstiene cuando debe 1.00


### Dónde se rompe: preguntas *casi* del dominio

Las preguntas de arriba eran obviamente ajenas (ceviche, Tokio). Las difíciles son las que **suenan** a tu corpus pero no están en él:

In [7]:
# ESQUEMA · CASI: preguntas «casi del dominio» (suenan a Andina Retail pero el dato no existe en el corpus).
# Son la prueba difícil: un umbral de similitud las deja pasar y el sistema respondería inventando.
CASI = [
    "¿Cuántos días de vacaciones da Google a sus empleados?",
    "¿Cuál es la garantía de la licuadora Philips HR2100?",
    "¿Qué significa ERR-9999?",
    "¿Cuál es la política de teletrabajo de Mercado Libre?",
]
filas = []
for q in CASI:
    d, b = señales(q)
    filas.append((q, round(d, 2), round(b, 1), "se abstiene" if (d < mejor[1] and b < mejor[2]) else "RESPONDE (alucinaría)"))
pd.DataFrame(filas, columns=["pregunta", "sim. densa máx.", "BM25 máx.", "decisión"])

,pregunta,sim. densa máx.,BM25 máx.,decisión
0,¿Cuántos días de vacaciones da Google a sus em...,0.57,5.2,RESPONDE (alucinaría)
1,¿Cuál es la garantía de la licuadora Philips H...,0.51,5.5,RESPONDE (alucinaría)
2,¿Qué significa ERR-9999?,0.49,0.0,RESPONDE (alucinaría)
3,¿Cuál es la política de teletrabajo de Mercado...,0.35,3.0,RESPONDE (alucinaría)


Un umbral de similitud **no distingue «parecido» de «respondido»**. Con preguntas casi del dominio la abstención falla; para esas hace falta
una verificación adicional (¿el pasaje recuperado **contiene** la respuesta? —un juez LLM o un modelo NLI—) y, sobre todo, citas obligatorias.

In [8]:
assert abst >= 0.8, "debería abstenerse en la mayoría de las preguntas fuera de alcance"
assert resp >= 0.8, "no debería abstenerse en la mayoría de preguntas que sí están en el corpus"
print("La abstención funciona en el set de prueba sin matar las respuestas válidas")

La abstención funciona en el set de prueba sin matar las respuestas válidas


## 3. Puerta de regresión

Antes de desplegar un cambio, se compara contra la configuración vigente en el **set de prueba**. Si empeora más de una tolerancia, el cambio no pasa: pasa solo si `MRR_candidato ≥ MRR_vigente − 0.03`.

⚠️ El «Cambio 1» (α = 0.8) pasa **por el borde** (0.86 frente a 0.89, con tolerancia 0.03): con otra tolerancia o más preguntas podría bloquearse. Es una señal de que el resultado es frágil, no una prueba de que 0.8 sea equivalente a 0.7.

In [9]:
_, test = dividir(QUERIES)


def puntaje(fn):
    return evaluar(fn, test).loc["PROMEDIO TIPOS", "MRR"]


def puerta(candidato, vigente, tolerancia=0.03):
    p_c, p_v = puntaje(candidato), puntaje(vigente)
    estado = "PASA" if p_c >= p_v - tolerancia else "BLOQUEADO"
    print(f"{estado:10s} candidato {p_c:.2f} vs vigente {p_v:.2f} (tolerancia {tolerancia})")
    return estado == "PASA"


vigente = lambda q: convexa(q, 0.7)
print("Cambio 1: subir el peso del denso a 0.8")
r1 = puerta(lambda q: convexa(q, 0.8), vigente)
print("Cambio 2: «simplificar» usando solo vectorial")
r2 = puerta(lambda q: r_denso(q), vigente)

Cambio 1: subir el peso del denso a 0.8


PASA       candidato 0.86 vs vigente 0.89 (tolerancia 0.03)
Cambio 2: «simplificar» usando solo vectorial


BLOQUEADO  candidato 0.73 vs vigente 0.89 (tolerancia 0.03)


In [10]:
assert r2 is False, "usar solo vectorial debería ser bloqueado por la puerta (pierde los códigos)"
print("La puerta bloquea el cambio que empeora la recuperación")

La puerta bloquea el cambio que empeora la recuperación


## 4. Latencia por método

Mediana (p50) y percentil 95 (p95) de la recuperación, incluyendo la vectorización de la pregunta. Los números dependen de tu máquina:
en Colab con CPU serán mayores. Lo importante es **medirlos en tu entorno** antes de comprometer un SLA.

In [11]:
def latencias(fn, preguntas, repeticiones=3):
    fn(preguntas[0])  # calentamiento
    t = []
    for _ in range(repeticiones):
        for q in preguntas:
            t0 = time.perf_counter(); fn(q); t.append((time.perf_counter() - t0) * 1000)
    return np.percentile(t, 50), np.percentile(t, 95)


preguntas = [q for q, _, _ in QUERIES]
lat = pd.DataFrame({
    "bm25": latencias(lambda q: r_bm25(q), preguntas),
    "denso": latencias(lambda q: r_denso(q), preguntas),
    "híbrido": latencias(lambda q: convexa(q, 0.7), preguntas),
}, index=["p50 (ms)", "p95 (ms)"]).T.round(2)
lat

,p50 (ms),p95 (ms)
bm25,0.03,0.04
denso,6.15,7.08
híbrido,6.23,7.44


In [12]:
assert lat.loc["bm25", "p50 (ms)"] < lat.loc["denso", "p50 (ms)"], "BM25 debería ser más rápido que vectorizar la pregunta"
print("BM25 es más barato; el híbrido paga el costo de vectorizar la pregunta")

BM25 es más barato; el híbrido paga el costo de vectorizar la pregunta


## Lista de verificación para llevarlo a producción

| Capa | Control | Dónde lo viste |
|---|---|---|
| Ingesta | Estructura y metadatos (rol, fuente, fecha) por documento | `RESTRINGIDOS`, notebook 02 |
| Seguridad | ACL antes del ranking + prueba de fuga | sección 1 |
| Recuperación | Híbrido calibrado con dev/test; router por tipo de pregunta | notebooks 01 y 04 |
| Fiabilidad | Abstención cuando no hay evidencia; fallback | sección 2 |
| Calidad | *Golden set* propio, puerta de regresión en CI | sección 3 |
| Operación | Latencia p50/p95, costo por consulta, trazas | sección 4 |

**Lo que este notebook no resuelve:** generación (el LLM), citas verificadas, actualización incremental del índice, ni evaluación con usuarios reales.
Son el siguiente paso, y todos se miden igual: con un set de preguntas con respuesta conocida.

---
*Webinar «RAG para Soluciones Productivas» · Manuel Argüelles · datos 100 % sintéticos de «Andina Retail»: las cifras ilustran un criterio de decisión, no miden tu sistema. Mide siempre con tus propios datos.*